# 🔍 Hybrid Retrieval: Combining Dense and Sparse Retrievers

**Course Reference:** [Ultimate RAG Bootcamp Using Langchain, LangGraph & Langsmith](https://www.udemy.com/course/ultimate-rag-bootcamp-using-langchainlanggraph-langsmith)

---

## 📚 Learning Objectives

By the end of this notebook, you will understand:
1. The difference between **Dense** and **Sparse** retrieval methods
2. How to implement a **BM25 (Sparse)** retriever for keyword-based search
3. How to implement a **FAISS (Dense)** retriever for semantic search
4. How to combine both using **EnsembleRetriever** for hybrid search
5. How to build a complete RAG pipeline with hybrid retrieval

---

## 🧠 Key Concepts

### What is Hybrid Search?

Hybrid search combines two complementary approaches:

| Approach | Method | Strengths | Weaknesses |
|----------|--------|-----------|------------|
| **Sparse (BM25)** | Keyword matching using TF-IDF | Exact keyword matches, handles rare terms well | Misses synonyms and semantic meaning |
| **Dense (Embeddings)** | Semantic similarity via vectors | Understands meaning, synonyms, context | May miss exact keyword matches |

### Why Combine Them?

- **BM25** excels when queries contain specific technical terms or exact phrases
- **Dense retrievers** excel at understanding intent and semantic similarity
- **Hybrid** gets the best of both worlds!

---

## 📦 Step 1: Import Required Libraries

We'll need the following components:
- **FAISS**: Facebook AI Similarity Search - a fast vector database for dense retrieval
- **HuggingFaceEmbeddings**: Pre-trained embedding models for creating dense vectors
- **BM25Retriever**: Sparse retriever using the BM25 ranking algorithm
- **EnsembleRetriever**: Combines multiple retrievers with weighted scoring

In [ ]:
# ============================================================================
# IMPORT LIBRARIES
# ============================================================================

# FAISS: A library for efficient similarity search and clustering of dense vectors
# Developed by Facebook AI Research, it's optimized for memory usage and speed
from langchain_community.vectorstores import FAISS

# HuggingFaceEmbeddings: Wrapper to use any HuggingFace sentence-transformer model
# These models convert text into dense vector representations (embeddings)
from langchain_huggingface import HuggingFaceEmbeddings

# BM25Retriever: Implements the Best Matching 25 (BM25) algorithm
# BM25 is a bag-of-words retrieval function that ranks documents based on 
# term frequency and inverse document frequency (TF-IDF variant)
from langchain_community.retrievers import BM25Retriever

# EnsembleRetriever: Combines multiple retrievers and merges their results
# Uses weighted scoring to balance contributions from each retriever
from langchain.retrievers import EnsembleRetriever

# Document: LangChain's document schema with page_content and metadata
from langchain.schema import Document

/Users/sourav.banerjee/Documents/Codebases/2. AI ENGINEERING/RAG_Demystified/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================================
# STEP 2: CREATE SAMPLE DOCUMENTS
# ============================================================================
# In a real-world scenario, these would come from document loaders (PDFs, web pages, etc.)
# For demonstration, we use simple text documents

docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# ============================================================================
# STEP 3: CREATE DENSE RETRIEVER (Semantic/Vector-based)
# ============================================================================
# Dense retrieval converts documents and queries into embedding vectors,
# then finds documents whose vectors are closest to the query vector.

# Initialize the embedding model
# "all-MiniLM-L6-v2" is a lightweight, fast model with 384-dimensional embeddings
# It's trained on 1B+ sentence pairs and works well for semantic similarity
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create FAISS vector store from documents
# This embeds all documents and stores them in an efficient index
dense_vectorstore = FAISS.from_documents(docs, embedding_model)

# Convert vector store to a retriever interface
# The retriever will return the most semantically similar documents
dense_retriever = dense_vectorstore.as_retriever()

print("✅ Dense retriever created successfully!")
print(f"   - Embedding model: all-MiniLM-L6-v2")
print(f"   - Vector store: FAISS")
print(f"   - Documents indexed: {len(docs)}")

In [ ]:
# ============================================================================
# STEP 4: CREATE SPARSE RETRIEVER (Keyword-based using BM25)
# ============================================================================
# BM25 (Best Matching 25) is a probabilistic ranking function that:
# - Scores documents based on term frequency (TF) - how often query terms appear
# - Considers inverse document frequency (IDF) - rare terms are weighted higher
# - Applies document length normalization - longer docs don't unfairly dominate
#
# Unlike dense retrieval, BM25 doesn't understand semantics but excels at:
# - Exact keyword matching
# - Finding rare/specific terms
# - Technical jargon and proper nouns

sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3  # Number of top documents to retrieve

print("✅ Sparse retriever (BM25) created successfully!")
print(f"   - Top-k documents: {sparse_retriever.k}")

# ============================================================================
# STEP 5: COMBINE INTO HYBRID/ENSEMBLE RETRIEVER
# ============================================================================
# The EnsembleRetriever merges results from multiple retrievers using:
# - Reciprocal Rank Fusion (RRF) or weighted scoring
# - Configurable weights to balance each retriever's contribution
#
# Weight distribution strategy:
# - weight=[0.7, 0.3] gives 70% importance to dense (semantic) and 30% to sparse (keyword)
# - Adjust based on your use case:
#   - More semantic queries → higher dense weight
#   - More keyword-specific queries → higher sparse weight
#   - Balanced → [0.5, 0.5]

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]  # 70% semantic, 30% keyword-based
)

print("\n✅ Hybrid (Ensemble) retriever created!")
print("   - Dense retriever weight: 0.7 (70%)")
print("   - Sparse retriever weight: 0.3 (30%)")


In [ ]:
# Inspect the hybrid retriever configuration
print("🔍 Hybrid Retriever Configuration:")
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3575a7d10>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x3ac0b5f90>, k=3)], weights=[0.5, 0.5])

In [ ]:
# ============================================================================
# STEP 6: TEST THE HYBRID RETRIEVER
# ============================================================================
# Let's test our hybrid retriever with a query that benefits from both approaches:
# - "build" and "application" → keyword match (BM25)
# - "LLMs" semantic meaning → dense retrieval understands the context

query = "How can I build an application using LLMs?"

print(f"📝 Query: \"{query}\"")
print("=" * 60)

# Retrieve relevant documents using the hybrid approach
results = hybrid_retriever.invoke(query)

# Display the retrieved results
print(f"\n📊 Retrieved {len(results)} documents:\n")
for i, doc in enumerate(results, 1):
    print(f"🔹 Document {i}:")
    print(f"   {doc.page_content}")
    print()

# ============================================================================
# 💡 OBSERVATION:
# Notice how the hybrid retriever found documents about:
# - Building LLM applications (semantic match)
# - Langchain development (keyword + semantic match)
# - Different types of retrievers (related context)
# 
# The irrelevant document about the Eiffel Tower was correctly excluded!
# ============================================================================


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


---

## 🤖 Part 2: Building a Complete RAG Pipeline with Hybrid Retrieval

Now that we have our hybrid retriever, let's integrate it into a full RAG (Retrieval-Augmented Generation) pipeline.

### RAG Pipeline Architecture:
```
Query → Hybrid Retriever → Retrieved Documents → LLM → Generated Answer
        (BM25 + Dense)      (Top-K docs)       (GPT-3.5)
```

### What We'll Build:

| Component | Purpose |
|-----------|---------|
| **Prompt Template** | Formats context + question for the LLM |
| **LLM** | Generates answers based on retrieved context |
| **Document Chain** | Combines retrieved docs into a single prompt |
| **RAG Chain** | End-to-end pipeline from query to answer |

### Why This Matters:

The LLM uses the retrieved documents as context to generate accurate, grounded responses. Without retrieval, LLMs:
- May hallucinate facts
- Can't access domain-specific knowledge
- Have knowledge cutoff dates

With hybrid RAG, we get **accurate, up-to-date, domain-specific answers**!

In [ ]:
# ============================================================================
# STEP 7: IMPORT RAG PIPELINE COMPONENTS
# ============================================================================

# init_chat_model: Universal initializer for various LLM providers
# Supports: "openai:gpt-4", "groq:llama-3.1-8b-instant", "anthropic:claude-3-haiku"
from langchain.chat_models import init_chat_model

# PromptTemplate: Creates reusable templates with variable substitution
# Variables are defined using {variable_name} syntax
from langchain.prompts import PromptTemplate

# create_stuff_documents_chain: Combines retrieved documents into LLM context
# "Stuff" strategy = concatenate all documents into a single prompt
from langchain.chains.combine_documents import create_stuff_documents_chain

# create_retrieval_chain: Connects retriever → documents → LLM → answer
from langchain.chains.retrieval import create_retrieval_chain

print("✅ RAG pipeline components imported!")

In [ ]:
# ============================================================================
# STEP 8: CREATE PROMPT TEMPLATE
# ============================================================================
# The prompt defines how we present context and question to the LLM
#
# Key variables:
# - {context}: Will be replaced with retrieved documents from hybrid retriever
# - {input}: Will be replaced with the user's question
#
# 💡 This simple prompt instructs the LLM to answer based ONLY on provided context

prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

print("✅ Prompt template created!")

# ============================================================================
# STEP 9: INITIALIZE THE LLM
# ============================================================================
# Using OpenAI's GPT-3.5-Turbo for answer generation
#
# Parameters:
# - temperature=0.2: Low value = focused, consistent answers
#   (0.0 = deterministic, 1.0 = more creative/random)
#
# 💡 Alternative models you can try:
# - "openai:gpt-4o-mini" - Better quality, slightly higher cost
# - "groq:llama-3.1-8b-instant" - Free, fast open-source option

llm = init_chat_model("openai:gpt-3.5-turbo", temperature=0.2)

print("✅ LLM initialized: GPT-3.5-Turbo (temperature=0.2)")
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x3a9454ed0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x3a9252850>, root_client=<openai.OpenAI object at 0x3a9264ad0>, root_async_client=<openai.AsyncOpenAI object at 0x3ac99e050>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [ ]:
# ============================================================================
# STEP 10: CREATE THE RAG CHAIN
# ============================================================================
# We combine all components into a single executable chain

# Create "stuff" document chain
# This chain takes retrieved documents and "stuffs" them into the prompt context
# It then sends the complete prompt to the LLM for answer generation
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

print("✅ Document chain created (stuff strategy)")

# Create the full RAG chain by connecting:
# 1. Hybrid Retriever (fetches relevant documents)
# 2. Document Chain (formats docs + generates answer)
#
# Pipeline: Query → Hybrid Retriever → Documents → Prompt → LLM → Answer
rag_chain = create_retrieval_chain(retriever=hybrid_retriever, combine_docs_chain=document_chain)

print("✅ Full RAG chain created!")
print("\n📊 Chain Flow:")
print("   Query → Hybrid Retriever → Retrieved Docs → Prompt + LLM → Answer")
rag_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3575a7d10>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x3ac0b5f90>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
            | ChatOpenAI(client=<openai.resour

In [ ]:
# ============================================================================
# STEP 11: TEST THE COMPLETE RAG PIPELINE
# ============================================================================
# Let's test our hybrid RAG system with a question about building LLM apps

# Define the query
# The hybrid retriever will use both:
# - BM25: To match keywords like "build", "app", "LLMs"
# - Dense: To understand semantic meaning of the question
query = {"input": "How can I build an app using LLMs?"}

print(f"🔍 Query: \"{query['input']}\"")
print("=" * 60)

# Invoke the RAG chain
# This triggers: Query → Hybrid Retrieval → LLM Generation
response = rag_chain.invoke(query)

# Display the generated answer
print("\n✅ ANSWER (Generated by LLM using hybrid-retrieved context):")
print("-" * 60)
print(response["answer"])

# Display the source documents that were retrieved
print("\n📄 SOURCE DOCUMENTS (Retrieved by Hybrid Search):")
print("-" * 60)
for i, doc in enumerate(response["context"]):
    print(f"\n🔹 Doc {i+1}: {doc.page_content}")

# ============================================================================
# 💡 OBSERVATION:
# Notice how the hybrid retriever found documents that are:
# - Semantically related to building applications (dense retrieval)
# - Keyword-matched with "LLM", "build", "application" (BM25)
# 
# The combination gives us more comprehensive context than either alone!
# ============================================================================

✅ Answer:
 You can build an app using LLMs by utilizing LangChain, which helps in developing LLM applications. LangChain can be used to develop agentic AI applications, and it offers various types of retrievers to enhance the functionality of your app. Additionally, you can also consider using Pinecone, a vector database for semantic search, to further optimize the performance of your LLM-based app.

📄 Source Documents:

Doc 1: LangChain helps build LLM applications.

Doc 2: Langchain can be used to develop agentic ai application.

Doc 3: Langchain has many types of retrievers.

Doc 4: Pinecone is a vector database for semantic search.


---

## 📚 Summary & Key Takeaways

### What We Learned:

1. **Sparse Retrieval (BM25)**: Keyword-based search using term frequency; great for exact matches
2. **Dense Retrieval (FAISS)**: Semantic search using embeddings; understands meaning and synonyms  
3. **Hybrid/Ensemble Retriever**: Combines both methods with configurable weights
4. **RAG Pipeline**: End-to-end system from Query → Retrieve → Generate

### When to Use Each Approach:

- **BM25 (Sparse)**: Technical documentation with specific terms
- **Dense (Semantic)**: Natural language questions
- **Hybrid**: Mixed queries with keywords + concepts ✅

### Next Steps:

- Experiment with different weights `[0.5, 0.5]` or `[0.3, 0.7]`
- Try different embedding models like `all-mpnet-base-v2`
- See `2-reranking.ipynb` for improving result quality
- See `3-mmr.ipynb` for diverse retrieval

🎉 **Congratulations!** You've built a complete hybrid search RAG system!
